# Music Annotation Project Notebook 59R3
## 151-Page 59R2-Retained Morphology and Failure-Family Audit

**Runtime:** CPU. Select **Runtime → Run all** in Google Colab.

This notebook answers:

> Across the 151-page annotated-only cohort, are the largest coherent 59R2-retained annotation-like regions dominated by one repeatable dark-ink morphology, or do pale near-threshold, dark-pen, top-right part/catalogue identifiers, long glyphs/arrows, and print-overlap candidates form distinct clusters requiring different treatments?

### Scientific boundary

This is an **annotated-only diagnostic**. It does not have clean truth and therefore cannot prove that every retained region is annotation. It identifies coherent retained candidate regions inside the authoritative reviewed mask, extracts observable morphology/context features, performs unsupervised clustering, and creates visual audit queues. Cluster names are descriptive hypotheses, not truth labels.

### Key additions

- analyses every scored coordinate from the completed 151-page 59R2 cohort;
- reconstructs observed pixel values from authoritative aligned pages when absent;
- identifies coherent retained regions at multiple 59R2 probability bands;
- explicitly tests recurring **top-right first-page identifiers** such as `D32`;
- explicitly tests **long glyphs / arrows / sweeping strokes**;
- compares unsupervised clusters against interpretable morphology flags;
- writes page, source-group, cluster and special-family summaries;
- produces representative crop sheets and whole-page locator sheets;
- keeps the 151-page cohort as failure-discovery/regression evidence, not supervised training truth.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib scikit-learn scikit-image


In [ ]:
# ============================================================
# Cell 2 - Configuration and output identity
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, platform, shutil, sys, zipfile

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from skimage.morphology import skeletonize

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
EXPECTED_PAGES = 151

# Optional exact overrides. Prefer these if automatic discovery is ambiguous.
NOTEBOOK64_RUN_OVERRIDE = None
SCORED_PARQUET_OVERRIDE = ROOT / '59r2_annotated_only_shadow_test' / '20261112_201411_59r2_shadow' / 'tables' / '59r2_scored_reviewed_cohort.parquet'
PAGE_RESOLUTION_CSV_OVERRIDE = None

# Retained-region definitions. p_remove is probability of removing annotation.
# A pixel is retained at threshold t when p_remove < t.
REFERENCE_THRESHOLD = 0.995
BANDS = [
    ('R1_0993_0995', 0.993, 0.995),
    ('R2_0985_0993', 0.985, 0.993),
    ('R3_0950_0985', 0.950, 0.985),
    ('R4_0900_0950', 0.900, 0.950),
    ('R5_LT_0900',   0.000, 0.900),
]
OBSERVED_DARK_CEILING = 200
MIN_REGION_PIXELS = 12
MORPH_CLOSE_RADIUS = 1
MAX_REGION_PIXELS_FOR_EXACT_SKELETON = 250000

# Positional hypotheses. These are diagnostic flags, never model inputs for promotion.
TOP_FRACTION = 0.24
RIGHT_FRACTION = 0.60
TOP_RIGHT_MAX_HEIGHT_FRACTION = 0.18
TOP_RIGHT_MAX_WIDTH_FRACTION = 0.30
LONG_ASPECT_MIN = 5.0
LONG_MAJOR_AXIS_FRACTION = 0.12

# Unsupervised model controls.
K_CANDIDATES = list(range(2, 9))
RANDOM_SEED = 42
REPRESENTATIVES_PER_CLUSTER = 18
SPECIAL_FAMILY_REPRESENTATIVES = 24
CROP_MARGIN = 45

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_notebook59R3_retained_morphology'
OUT = ROOT / 'notebook59R3_retained_morphology' / RUN_ID
TABLES = OUT / 'tables'
CROPS = OUT / 'crops'
SHEETS = OUT / 'contact_sheets'
WHOLE = OUT / 'whole_page_locators'
AUDIT = OUT / 'audit'
UPLOAD = OUT / 'upload'
for folder in [OUT, TABLES, CROPS, SHEETS, WHOLE, AUDIT, UPLOAD]:
    folder.mkdir(parents=True, exist_ok=False)

print('Output:', OUT)


In [ ]:
# ============================================================
# Cell 3 - Helpers and source-run resolution
# ============================================================

SUPPORTED = {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()

def atomic_json(path, value):
    tmp = Path(str(path) + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, default=str))
    tmp.replace(path)

def array_hash(a):
    a = np.ascontiguousarray(a)
    h = hashlib.sha256()
    h.update(str(a.dtype).encode())
    h.update(str(a.shape).encode())
    h.update(a.tobytes())
    return h.hexdigest()

def latest_complete(root, required):
    hits=[]
    root=Path(root)
    if root.exists():
        for p in root.rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required):
                hits.append(p)
    return max(hits, key=lambda p:p.stat().st_mtime) if hits else None

def resolve_notebook64_run():
    if NOTEBOOK64_RUN_OVERRIDE is not None:
        run=Path(NOTEBOOK64_RUN_OVERRIDE)
        require(run.is_dir(), f'Notebook64 override does not exist: {run}')
        return run
    roots=[
        ROOT/'notebook64_annotated_only_threshold_visual_audit',
        ROOT/'notebook64_exact_truth_and_threshold_safety',
        ROOT/'notebook64_threshold_visual_audit',
        ROOT/'Notebook64',
    ]
    candidates=[]
    for root in roots:
        if root.exists():
            for p in root.rglob('*'):
                if p.is_dir() and (p/'audit'/'page_pair_resolution.csv').exists():
                    candidates.append(p)
    require(candidates, 'Could not find a Notebook64 run. Set NOTEBOOK64_RUN_OVERRIDE.')
    return max(candidates, key=lambda p:p.stat().st_mtime)

def find_scored_parquet(run):
    if SCORED_PARQUET_OVERRIDE is not None:
        path=Path(SCORED_PARQUET_OVERRIDE)
        require(path.is_file(), f'Scored parquet override not found: {path}')
        return path
    likely=[]
    for root in [run, ROOT/'observed_print_classifier_v8_2', ROOT/'notebook61', ROOT]:
        if root.exists():
            for pattern in ['**/*scored*.parquet','**/*59r2*.parquet','**/all_eligible_oof_predictions.parquet']:
                likely.extend(root.glob(pattern))
    good=[]
    for path in sorted(set(likely), key=lambda p:p.stat().st_mtime, reverse=True):
        try:
            cols=set(pd.read_parquet(path, columns=None).columns)
            if {'page','y','x'}.issubset(cols) and ({'p_remove_59r2','p_remove'} & cols):
                good.append(path)
        except Exception:
            continue
    require(good, 'Could not resolve the 59R2 scored parquet. Set SCORED_PARQUET_OVERRIDE.')
    return good[0]

def parse_path(value):
    if pd.isna(value): return None
    p=Path(str(value))
    return p if p.exists() else None

NB64 = resolve_notebook64_run()
PARQUET_PATH = find_scored_parquet(NB64)
PAIR_CSV = Path(PAGE_RESOLUTION_CSV_OVERRIDE) if PAGE_RESOLUTION_CSV_OVERRIDE else NB64/'audit'/'page_pair_resolution.csv'
require(PAIR_CSV.is_file(), f'Missing page resolution CSV: {PAIR_CSV}')

print('Notebook64:', NB64)
print('Scored parquet:', PARQUET_PATH)
print('Page resolution:', PAIR_CSV)


In [ ]:
# ============================================================
# Cell 4 - Load the exact 151-page cohort and observed pages
# ============================================================

cohort = pd.read_parquet(PARQUET_PATH)
if 'p_remove_59r2' not in cohort.columns and 'p_remove' in cohort.columns:
    cohort = cohort.rename(columns={'p_remove':'p_remove_59r2'})
required={'page','y','x','p_remove_59r2'}
missing=required-set(cohort.columns)
require(not missing, f'Scored parquet missing columns: {sorted(missing)}')
cohort['page']=cohort.page.astype(str)
cohort['y']=pd.to_numeric(cohort.y, errors='raise').astype(np.int32)
cohort['x']=pd.to_numeric(cohort.x, errors='raise').astype(np.int32)
cohort['p_remove_59r2']=pd.to_numeric(cohort.p_remove_59r2, errors='raise').astype(np.float64)
require(np.isfinite(cohort.p_remove_59r2).all(), 'Non-finite p_remove_59r2 values detected.')
require(not cohort.duplicated(['page','y','x']).any(), 'Duplicate page/y/x rows detected.')
require(cohort.page.nunique()==EXPECTED_PAGES, f'Expected {EXPECTED_PAGES} pages, found {cohort.page.nunique()}.')

resolution=pd.read_csv(PAIR_CSV)
require('page' in resolution.columns, 'page_pair_resolution.csv lacks page.')
resolution['page']=resolution.page.astype(str)

observed_column=next((c for c in ['observed_path','observed','aligned_output','resolved_observed_path'] if c in resolution.columns),None)
require(observed_column is not None, f'Could not identify observed-path column in {list(resolution.columns)}')
observed_paths={str(r.page):parse_path(r[observed_column]) for _,r in resolution.iterrows()}
missing_pages=sorted(set(cohort.page.unique())-{p for p,v in observed_paths.items() if v is not None})
require(not missing_pages, f'Observed pages unresolved: {missing_pages[:20]}')

page_runtime={}
observed_values=np.empty(len(cohort), dtype=np.uint8)
for page,idx in cohort.groupby('page',sort=False).groups.items():
    idx=np.asarray(list(idx),dtype=np.int64)
    obs=cv2.imread(str(observed_paths[page]),cv2.IMREAD_GRAYSCALE)
    require(obs is not None, f'{page}: unreadable observed page {observed_paths[page]}')
    y=cohort.loc[idx,'y'].to_numpy(np.int32); x=cohort.loc[idx,'x'].to_numpy(np.int32)
    require((y>=0).all() and (x>=0).all() and (y<obs.shape[0]).all() and (x<obs.shape[1]).all(), f'{page}: coordinates outside observed page')
    observed_values[idx]=obs[y,x]
    page_runtime[page]={'observed':obs,'path':observed_paths[page]}
cohort['observed_value']=observed_values

# Optional fields are normalised when present.
if 'source_group' not in cohort.columns:
    cohort['source_group']='UNKNOWN'
if 'component_key' not in cohort.columns:
    cohort['component_key']='UNAVAILABLE'

identity={
    'notebook64_run':str(NB64),
    'scored_parquet':str(PARQUET_PATH),
    'scored_parquet_sha256':sha256_file(PARQUET_PATH),
    'page_resolution_csv':str(PAIR_CSV),
    'page_resolution_sha256':sha256_file(PAIR_CSV),
    'pages':int(cohort.page.nunique()),
    'rows':int(len(cohort)),
    'observed_value_reconstructed':True,
    'clean_truth_used':False,
}
atomic_json(AUDIT/'input_identity.json',identity)
print(json.dumps(identity,indent=2))


In [ ]:
# ============================================================
# Cell 5 - Build coherent retained regions in probability bands
# ============================================================

# Regions are connected components of eligible dark coordinates in each p_remove band.
# This avoids merging a whole broad mask component containing several distinct marks.

region_rows=[]
region_pixel_rows=[]
region_masks={}

for page,g in cohort.groupby('page',sort=False):
    obs=page_runtime[page]['observed']; H,W=obs.shape
    ys=g.y.to_numpy(np.int32); xs=g.x.to_numpy(np.int32); probs=g.p_remove_59r2.to_numpy(); vals=g.observed_value.to_numpy()
    for band_id,low,high in BANDS:
        selected=(probs>=low)&(probs<high)&(vals<OBSERVED_DARK_CEILING)
        if not selected.any(): continue
        binary=np.zeros((H,W),np.uint8); binary[ys[selected],xs[selected]]=1
        if MORPH_CLOSE_RADIUS>0:
            k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*MORPH_CLOSE_RADIUS+1,2*MORPH_CLOSE_RADIUS+1))
            connected=cv2.morphologyEx(binary,cv2.MORPH_CLOSE,k)
            connected &= (binary>0).astype(np.uint8)  # retain exact scored pixels after connectivity aid
            # Components must be computed on bridged support, then intersect exact pixels below.
            bridge=cv2.morphologyEx(binary,cv2.MORPH_CLOSE,k)
        else:
            bridge=binary
        n,labels,stats,cent=cv2.connectedComponentsWithStats(bridge,8)
        exact_label=labels[ys[selected],xs[selected]]
        selected_indices=g.index.to_numpy()[selected]
        for cid in range(1,n):
            member=exact_label==cid
            exact_count=int(member.sum())
            if exact_count<MIN_REGION_PIXELS: continue
            global_idx=selected_indices[member]
            q=cohort.loc[global_idx]
            x=int(q.x.min()); y=int(q.y.min()); x1=int(q.x.max())+1; y1=int(q.y.max())+1
            w=x1-x; h=y1-y; area=exact_count
            rid=f'{page}__{band_id}__r{cid:04d}'
            region_rows.append({'region_id':rid,'page':page,'source_group':str(q.source_group.iloc[0]),'band_id':band_id,'band_low':low,'band_high':high,'pixels':area,'x':x,'y':y,'w':w,'h':h,'cx':float(q.x.mean()),'cy':float(q.y.mean()),'page_w':W,'page_h':H})
            region_pixel_rows.extend({'region_id':rid,'page':page,'y':int(yy),'x':int(xx),'p_remove_59r2':float(pp),'observed_value':int(vv)} for yy,xx,pp,vv in zip(q.y,q.x,q.p_remove_59r2,q.observed_value))

regions=pd.DataFrame(region_rows)
region_pixels=pd.DataFrame(region_pixel_rows)
require(len(regions)>0,'No retained coherent regions were created.')
regions.to_csv(TABLES/'retained_region_geometry.csv',index=False)
region_pixels.to_parquet(TABLES/'retained_region_pixels.parquet',index=False)
print('Regions:',len(regions),'Pages:',regions.page.nunique(),'Pixels:',len(region_pixels))
print(regions.band_id.value_counts())


In [ ]:
# ============================================================
# Cell 6 - Observable morphology, topology and context features
# ============================================================

feature_rows=[]
for _,r in regions.iterrows():
    page=r.page; obs=page_runtime[page]['observed']; H,W=obs.shape
    q=region_pixels[region_pixels.region_id==r.region_id]
    x0,y0,x1,y1=int(r.x),int(r.y),int(r.x+r.w),int(r.y+r.h)
    local=np.zeros((int(r.h),int(r.w)),bool)
    local[q.y.to_numpy(int)-y0,q.x.to_numpy(int)-x0]=True
    vals=q.observed_value.to_numpy(np.float32); probs=q.p_remove_59r2.to_numpy(np.float64)
    bbox_area=max(int(r.w*r.h),1); fill=float(local.sum()/bbox_area)
    aspect=max(float(r.w),float(r.h))/max(min(float(r.w),float(r.h)),1.0)

    # Skeleton/shape descriptors.
    if local.size<=MAX_REGION_PIXELS_FOR_EXACT_SKELETON:
        sk=skeletonize(local)
        skeleton_pixels=int(sk.sum())
        # endpoint count under 8-connectivity.
        neigh=cv2.filter2D(sk.astype(np.uint8),-1,np.ones((3,3),np.uint8)) - sk.astype(np.uint8)
        endpoints=int(np.sum(sk & (neigh==1)))
        junctions=int(np.sum(sk & (neigh>=3)))
    else:
        skeleton_pixels=np.nan; endpoints=np.nan; junctions=np.nan

    # Orientation from coordinate covariance.
    coords=np.column_stack([q.x.to_numpy(float),q.y.to_numpy(float)])
    if len(coords)>=2:
        cov=np.cov(coords,rowvar=False); eigvals,eigvecs=np.linalg.eigh(cov); order=np.argsort(eigvals)[::-1]; eigvals=eigvals[order]; vec=eigvecs[:,order[0]]
        major=float(math.sqrt(max(eigvals[0],0))*4); minor=float(math.sqrt(max(eigvals[1],0))*4)
        angle=float((math.degrees(math.atan2(vec[1],vec[0]))+180)%180)
        linearity=float(eigvals[0]/max(eigvals.sum(),1e-9))
    else:
        major=minor=angle=0.0; linearity=0.0

    # Local context ring around exact region.
    pad=18; yy0=max(0,y0-pad); xx0=max(0,x0-pad); yy1=min(H,y1+pad); xx1=min(W,x1+pad)
    patch=obs[yy0:yy1,xx0:xx1]
    region_patch=np.zeros(patch.shape,np.uint8); region_patch[q.y.to_numpy(int)-yy0,q.x.to_numpy(int)-xx0]=1
    ring=cv2.dilate(region_patch,np.ones((37,37),np.uint8))>0
    ring &= ~(region_patch>0)
    edge=cv2.Canny(patch,70,170)>0
    context_dark=float((patch[ring]<190).mean()) if ring.any() else 0.0
    context_edge=float(edge[ring].mean()) if ring.any() else 0.0

    top_right=bool((r.cy/H)<=TOP_FRACTION and (r.cx/W)>=RIGHT_FRACTION and (r.h/H)<=TOP_RIGHT_MAX_HEIGHT_FRACTION and (r.w/W)<=TOP_RIGHT_MAX_WIDTH_FRACTION)
    long_glyph=bool(aspect>=LONG_ASPECT_MIN or major/max(H,W)>=LONG_MAJOR_AXIS_FRACTION)
    pale_near=bool(np.median(vals)>=145 and np.median(probs)>=0.985)
    dark_pen=bool(np.median(vals)<110 and np.quantile(probs,.75)<0.985)
    dense_dark=bool(np.quantile(vals,.75)<130 and fill>=0.08)

    feature_rows.append({
        'region_id':r.region_id,'page':page,'source_group':r.source_group,'band_id':r.band_id,
        'pixels':int(r.pixels),'x':x0,'y':y0,'w':int(r.w),'h':int(r.h),'cx':r.cx,'cy':r.cy,'page_w':W,'page_h':H,
        'x_fraction':r.cx/W,'y_fraction':r.cy/H,'w_fraction':r.w/W,'h_fraction':r.h/H,
        'bbox_fill':fill,'aspect_ratio':aspect,'major_axis_proxy':major,'minor_axis_proxy':minor,'orientation_deg':angle,'linearity':linearity,
        'skeleton_pixels':skeleton_pixels,'skeleton_to_pixels':skeleton_pixels/max(int(r.pixels),1) if np.isfinite(skeleton_pixels) else np.nan,'endpoints':endpoints,'junctions':junctions,
        'observed_mean':float(vals.mean()),'observed_median':float(np.median(vals)),'observed_q25':float(np.quantile(vals,.25)),'observed_q75':float(np.quantile(vals,.75)),'observed_std':float(vals.std()),
        'p_remove_mean':float(probs.mean()),'p_remove_median':float(np.median(probs)),'p_remove_min':float(probs.min()),'p_remove_max':float(probs.max()),'p_remove_std':float(probs.std()),
        'context_dark_density':context_dark,'context_edge_density':context_edge,
        'flag_top_right_identifier_candidate':top_right,'flag_long_glyph_candidate':long_glyph,'flag_pale_near_threshold':pale_near,'flag_dark_pen_candidate':dark_pen,'flag_dense_dark_candidate':dense_dark,
    })

features=pd.DataFrame(feature_rows)
features.to_csv(TABLES/'retained_region_features.csv',index=False)
print(features[['flag_top_right_identifier_candidate','flag_long_glyph_candidate','flag_pale_near_threshold','flag_dark_pen_candidate']].sum())


In [ ]:
# ============================================================
# Cell 7 - Unsupervised clustering and cluster selection
# ============================================================

MODEL_FEATURES=[
    'pixels','w_fraction','h_fraction','bbox_fill','aspect_ratio','major_axis_proxy','minor_axis_proxy','linearity',
    'skeleton_to_pixels','endpoints','junctions','observed_mean','observed_median','observed_std',
    'p_remove_mean','p_remove_median','p_remove_std','context_dark_density','context_edge_density'
]
X=features[MODEL_FEATURES].replace([np.inf,-np.inf],np.nan)
pre=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',RobustScaler())])
Z=pre.fit_transform(X)

scores=[]; models={}
for k in K_CANDIDATES:
    if len(features)<=k: continue
    model=KMeans(n_clusters=k,random_state=RANDOM_SEED,n_init=30)
    labels=model.fit_predict(Z)
    score=float(silhouette_score(Z,labels)) if len(set(labels))>1 else np.nan
    scores.append({'k':k,'silhouette':score,'inertia':float(model.inertia_)})
    models[k]=(model,labels)
selection=pd.DataFrame(scores)
require(len(selection)>0,'Insufficient regions for clustering.')
best_k=int(selection.sort_values(['silhouette','k'],ascending=[False,True]).iloc[0].k)
model,labels=models[best_k]
features['cluster_id']=labels
selection.to_csv(TABLES/'cluster_model_selection.csv',index=False)

profile=features.groupby('cluster_id',as_index=False).agg(
    regions=('region_id','size'),pages=('page','nunique'),source_groups=('source_group','nunique'),pixels=('pixels','sum'),
    median_pixels=('pixels','median'),median_observed=('observed_median','median'),median_p_remove=('p_remove_median','median'),
    median_aspect=('aspect_ratio','median'),median_linearity=('linearity','median'),median_fill=('bbox_fill','median'),
    top_right_rate=('flag_top_right_identifier_candidate','mean'),long_glyph_rate=('flag_long_glyph_candidate','mean'),
    pale_rate=('flag_pale_near_threshold','mean'),dark_pen_rate=('flag_dark_pen_candidate','mean'),dense_dark_rate=('flag_dense_dark_candidate','mean'),
    median_context_dark=('context_dark_density','median'),median_context_edge=('context_edge_density','median')
)

def cluster_hypothesis(row):
    candidates=[]
    if row.top_right_rate>=0.30: candidates.append('TOP_RIGHT_IDENTIFIER_HEAVY')
    if row.long_glyph_rate>=0.35: candidates.append('LONG_GLYPH_STROKE_HEAVY')
    if row.pale_rate>=0.40: candidates.append('PALE_NEAR_THRESHOLD_HEAVY')
    if row.dark_pen_rate>=0.35 or row.dense_dark_rate>=0.45: candidates.append('DARK_PEN_DENSE_HEAVY')
    if row.median_context_edge>=0.12 or row.median_context_dark>=0.18: candidates.append('PRINT_CONTEXT_OVERLAP_HEAVY')
    return '|'.join(candidates) if candidates else 'MIXED_OR_OTHER'
profile['descriptive_hypothesis']=profile.apply(cluster_hypothesis,axis=1)
profile.to_csv(TABLES/'cluster_profiles.csv',index=False)
features.to_csv(TABLES/'retained_regions_clustered.csv',index=False)
print('Selected k:',best_k)
display(selection)
display(profile)


In [ ]:
# ============================================================
# Cell 8 - Special-family prevalence and part-number recurrence
# ============================================================

family_defs={
    'TOP_RIGHT_IDENTIFIER_CANDIDATE':'flag_top_right_identifier_candidate',
    'LONG_GLYPH_CANDIDATE':'flag_long_glyph_candidate',
    'PALE_NEAR_THRESHOLD':'flag_pale_near_threshold',
    'DARK_PEN_CANDIDATE':'flag_dark_pen_candidate',
    'DENSE_DARK_CANDIDATE':'flag_dense_dark_candidate',
}
family_rows=[]
for family,col in family_defs.items():
    q=features[features[col]]
    family_rows.append({'family':family,'regions':len(q),'pages':q.page.nunique(),'source_groups':q.source_group.nunique(),'pixels':int(q.pixels.sum()),'median_observed':float(q.observed_median.median()) if len(q) else np.nan,'median_p_remove':float(q.p_remove_median.median()) if len(q) else np.nan,'median_aspect':float(q.aspect_ratio.median()) if len(q) else np.nan})
family_summary=pd.DataFrame(family_rows)
family_summary.to_csv(TABLES/'special_family_summary.csv',index=False)

# Page-level top-right burden is useful for repeated first-page part/catalogue IDs such as D32.
page_special=features.groupby(['page','source_group'],as_index=False).agg(
    regions=('region_id','size'),retained_pixels=('pixels','sum'),
    top_right_regions=('flag_top_right_identifier_candidate','sum'),long_glyph_regions=('flag_long_glyph_candidate','sum'),
    pale_regions=('flag_pale_near_threshold','sum'),dark_pen_regions=('flag_dark_pen_candidate','sum')
)
page_special.to_csv(TABLES/'special_families_by_page.csv',index=False)

# Recurrence fingerprint: coarse normalised position/size/morphology bins, useful for repeated D32-like placements.
top=features[features.flag_top_right_identifier_candidate].copy()
if len(top):
    top['position_bin']=pd.cut(top.x_fraction,[0,.6,.7,.8,.9,1.001],include_lowest=True).astype(str)+'|'+pd.cut(top.y_fraction,[0,.05,.10,.15,.20,.25],include_lowest=True).astype(str)
    top['size_bin']=pd.cut(top.pixels,[0,25,75,200,600,2000,np.inf],include_lowest=True).astype(str)
    top['shape_bin']=pd.cut(top.aspect_ratio,[0,1.5,3,6,12,np.inf],include_lowest=True).astype(str)
    recurrence=top.groupby(['position_bin','size_bin','shape_bin'],as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'),source_groups=('source_group','nunique'),median_observed=('observed_median','median'),median_p_remove=('p_remove_median','median')).sort_values(['pages','regions'],ascending=False)
else:
    recurrence=pd.DataFrame(columns=['position_bin','size_bin','shape_bin','regions','pages','source_groups','median_observed','median_p_remove'])
recurrence.to_csv(TABLES/'top_right_recurrence_fingerprints.csv',index=False)

display(family_summary)
display(recurrence.head(20))


In [ ]:
# ============================================================
# Cell 9 - Representative crops and contact sheets
# ============================================================


def crop_region(row,margin=CROP_MARGIN):
    obs=page_runtime[row.page]['observed']; H,W=obs.shape
    x0=max(0,int(row.x)-margin); y0=max(0,int(row.y)-margin); x1=min(W,int(row.x+row.w)+margin); y1=min(H,int(row.y+row.h)+margin)
    crop=obs[y0:y1,x0:x1]
    rgb=cv2.cvtColor(crop,cv2.COLOR_GRAY2BGR)
    cv2.rectangle(rgb,(int(row.x)-x0,int(row.y)-y0),(int(row.x+row.w)-x0,int(row.y+row.h)-y0),(0,0,255),2)
    return rgb

def save_region_crop(row,folder,prefix):
    folder.mkdir(parents=True,exist_ok=True)
    im=crop_region(row)
    path=folder/f'{prefix}_{row.page}_{row.region_id.split("__")[-1]}.png'
    cv2.imwrite(str(path),im)
    return path

def make_contact(rows,name,title,max_items):
    rows=rows.head(max_items).reset_index(drop=True)
    if len(rows)==0:return None
    cols=4; rcount=math.ceil(len(rows)/cols)
    fig,axes=plt.subplots(rcount,cols,figsize=(16,4.2*rcount),squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for i,row in rows.iterrows():
        im=cv2.cvtColor(crop_region(row),cv2.COLOR_BGR2RGB)
        ax=axes.flat[i]; ax.imshow(im); ax.set_title(f'{row.page} | C{int(row.cluster_id)} | {row.band_id}
px={int(row.pixels)} obs={row.observed_median:.0f} p={row.p_remove_median:.3f} asp={row.aspect_ratio:.1f}',fontsize=9); ax.axis('off')
    fig.suptitle(title,fontsize=15); plt.tight_layout(); path=SHEETS/f'{name}.png'; plt.savefig(path,dpi=160,bbox_inches='tight'); plt.close(fig); return path

# Cluster representatives: prefer large regions but retain diversity by pages.
for cid,g in features.groupby('cluster_id'):
    reps=g.sort_values(['pixels','p_remove_median'],ascending=[False,True]).drop_duplicates('page').head(REPRESENTATIVES_PER_CLUSTER)
    make_contact(reps,f'cluster_{int(cid):02d}_representatives',f'Cluster {cid} representatives',REPRESENTATIVES_PER_CLUSTER)

for family,col in family_defs.items():
    reps=features[features[col]].sort_values(['pixels','p_remove_median'],ascending=[False,True]).drop_duplicates('page').head(SPECIAL_FAMILY_REPRESENTATIVES)
    make_contact(reps,f'family_{family.lower()}',family.replace('_',' '),SPECIAL_FAMILY_REPRESENTATIVES)

print('Contact sheets:',SHEETS)


In [ ]:
# ============================================================
# Cell 10 - Whole-page locators for top-right and long-glyph families
# ============================================================

locator_rows=[]
for page,g in features.groupby('page'):
    special=g[g.flag_top_right_identifier_candidate | g.flag_long_glyph_candidate]
    if len(special)==0: continue
    obs=page_runtime[page]['observed']; rgb=cv2.cvtColor(obs,cv2.COLOR_GRAY2BGR)
    for _,r in special.iterrows():
        if r.flag_top_right_identifier_candidate and r.flag_long_glyph_candidate: colour=(255,0,255); label='TR+LONG'
        elif r.flag_top_right_identifier_candidate: colour=(0,0,255); label='TOP-RIGHT'
        else: colour=(255,120,0); label='LONG'
        cv2.rectangle(rgb,(int(r.x),int(r.y)),(int(r.x+r.w),int(r.y+r.h)),colour,3)
        cv2.putText(rgb,label,(int(r.x),max(18,int(r.y)-5)),cv2.FONT_HERSHEY_SIMPLEX,.55,colour,2,cv2.LINE_AA)
    scale=min(1.0,1600/max(rgb.shape)); preview=cv2.resize(rgb,None,fx=scale,fy=scale,interpolation=cv2.INTER_AREA) if scale<1 else rgb
    path=WHOLE/f'{page}_special_family_locator.png'; cv2.imwrite(str(path),preview)
    locator_rows.append({'page':page,'regions':len(special),'top_right':int(special.flag_top_right_identifier_candidate.sum()),'long_glyph':int(special.flag_long_glyph_candidate.sum()),'path':str(path)})
pd.DataFrame(locator_rows).to_csv(TABLES/'whole_page_locator_inventory.csv',index=False)
print('Whole-page locators:',len(locator_rows))


In [ ]:
# ============================================================
# Cell 11 - Decision-oriented synthesis
# ============================================================

# Determine whether one cluster dominates retained pixels and pages.
cluster_share=profile.copy()
cluster_share['pixel_share']=cluster_share.pixels/cluster_share.pixels.sum()
cluster_share['region_share']=cluster_share.regions/cluster_share.regions.sum()
cluster_share['page_share']=cluster_share.pages/EXPECTED_PAGES
cluster_share.to_csv(TABLES/'cluster_prevalence.csv',index=False)

dominant=cluster_share.sort_values('pixel_share',ascending=False).iloc[0]
family_page_rates={row.family:float(row.pages/EXPECTED_PAGES) for _,row in family_summary.iterrows()}

# Diagnostic decision, not scientific truth.
if dominant.pixel_share>=0.70 and dominant.page_share>=0.70:
    conclusion='ONE_DOMINANT_REPEATABLE_MORPHOLOGY'
elif (cluster_share.pixel_share>=0.15).sum()>=2 or sum(v>=0.15 for v in family_page_rates.values())>=2:
    conclusion='MULTIPLE_DISTINCT_FAILURE_FAMILIES'
else:
    conclusion='MIXED_EVIDENCE_NO_SINGLE_DOMINANT_FAMILY'

# Treatment implications are intentionally separated.
treatment_rows=[]
for _,row in family_summary.iterrows():
    if row.family=='PALE_NEAR_THRESHOLD': treatment='Calibration/operating-point research may help, but only with safety scoring.'
    elif row.family=='DARK_PEN_CANDIDATE': treatment='Representation/learning problem: add authoritative dark-ink annotation evidence to a spatial challenger.'
    elif row.family=='TOP_RIGHT_IDENTIFIER_CANDIDATE': treatment='Add page-role/context experiment or larger-context branch; do not use absolute coordinates as a production shortcut without validation.'
    elif row.family=='LONG_GLYPH_CANDIDATE': treatment='Add larger/multi-scale topology or stroke-continuation representation; 33x33 context may truncate the glyph.'
    elif row.family=='DENSE_DARK_CANDIDATE': treatment='Test dark dense handwriting separately from pale pencil and genuine print-overlap.'
    else: treatment='Review.'
    treatment_rows.append({'family':row.family,'regions':int(row.regions),'pages':int(row.pages),'page_fraction':float(row.pages/EXPECTED_PAGES),'treatment_hypothesis':treatment})
treatments=pd.DataFrame(treatment_rows)
treatments.to_csv(TABLES/'failure_family_treatment_hypotheses.csv',index=False)

result={
    'conclusion':conclusion,
    'selected_clusters':best_k,
    'regions':int(len(features)),
    'pages_with_regions':int(features.page.nunique()),
    'dominant_cluster':int(dominant.cluster_id),
    'dominant_cluster_pixel_share':float(dominant.pixel_share),
    'dominant_cluster_page_share':float(dominant.page_share),
    'family_page_rates':family_page_rates,
    'clean_truth_available':False,
    'cluster_names_are_hypotheses':True,
    'promotion_authority':False,
}
atomic_json(OUT/'notebook59R3_result.json',result)
print(json.dumps(result,indent=2))
display(cluster_share.sort_values('pixel_share',ascending=False))
display(treatments)


In [ ]:
# ============================================================
# Cell 12 - Integrity verdict, summary and curated upload bundle
# ============================================================

checks={
    '151_page_roster':cohort.page.nunique()==EXPECTED_PAGES,
    'unique_page_y_x':not cohort.duplicated(['page','y','x']).any(),
    'all_observed_pages_resolved':len(page_runtime)==EXPECTED_PAGES,
    'observed_values_complete':cohort.observed_value.notna().all(),
    'retained_regions_nonempty':len(features)>0,
    'every_region_has_pixels':bool((features.pixels>=MIN_REGION_PIXELS).all()),
    'cluster_assignment_complete':features.cluster_id.notna().all(),
    'cluster_profiles_written':(TABLES/'cluster_profiles.csv').is_file(),
    'special_family_summary_written':(TABLES/'special_family_summary.csv').is_file(),
    'no_clean_truth_used':True,
}
first_failure=next((k for k,v in checks.items() if not v),None)
verdict='PASS' if first_failure is None else 'FAIL'
verdict_record={'verdict':verdict,'first_failed_check':first_failure,'checks':checks,'created_utc':datetime.now(timezone.utc).isoformat(),'promotion_authority':False}
atomic_json(AUDIT/'notebook59R3_verdict.json',verdict_record)
require(verdict=='PASS',f'Notebook59R3 failed: {first_failure}')

env={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
atomic_json(OUT/'software_environment.json',env)

lines=[
    'NOTEBOOK 59R3 - 151-PAGE RETAINED MORPHOLOGY AUDIT','',
    f'VERDICT: {verdict}',f'Conclusion: {conclusion}',f'Pages: {cohort.page.nunique()}',f'Scored pixels: {len(cohort):,}',f'Coherent retained regions: {len(features):,}',f'Unsupervised clusters selected: {best_k}',
    f'Dominant cluster pixel share: {dominant.pixel_share:.4f}',f'Dominant cluster page share: {dominant.page_share:.4f}','',
    'SPECIAL FAMILIES'
]
for _,r in family_summary.iterrows():
    lines.append(f"- {r.family}: {int(r.regions):,} regions across {int(r.pages)} pages ({r.pages/EXPECTED_PAGES:.1%})")
lines += ['', 'INTERPRETATION',
    'This is an annotated-only failure-family audit. Region clusters are observable morphology hypotheses, not clean-truth labels.',
    'Top-right placement is explicitly analysed because first-page part/catalogue identifiers such as D32 recur in that location.',
    'Long-glyph candidates are explicitly analysed because arrows and sweeping strokes may exceed the 33x33 local context used by 59R2 lineage models.',
    'Different families should not be forced into one threshold treatment: pale near-threshold material, dark dense pen, top-right identifiers, long glyphs and print-overlap candidates can require different model/context changes.',
    'No result authorises retraining, threshold promotion or production use without authoritative labels and fresh validation.'
]
(OUT/'executive_summary.txt').write_text('
'.join(lines)+'
')
print('
'.join(lines))

# Curated package.
curated=[
    OUT/'executive_summary.txt',OUT/'notebook59R3_result.json',OUT/'software_environment.json',
    AUDIT/'input_identity.json',AUDIT/'notebook59R3_verdict.json',
    TABLES/'cluster_model_selection.csv',TABLES/'cluster_profiles.csv',TABLES/'cluster_prevalence.csv',
    TABLES/'retained_region_features.csv',TABLES/'retained_regions_clustered.csv',
    TABLES/'special_family_summary.csv',TABLES/'special_families_by_page.csv',
    TABLES/'top_right_recurrence_fingerprints.csv',TABLES/'failure_family_treatment_hypotheses.csv',
    TABLES/'whole_page_locator_inventory.csv',
] + sorted(SHEETS.glob('*.png'))

for src in curated:
    require(src.is_file(),f'Missing curated artefact: {src}')
    rel=src.relative_to(OUT); dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst)
    require(sha256_file(src)==sha256_file(dst),f'Copy hash mismatch: {rel}')

(UPLOAD/'README.txt').write_text('Notebook 59R3 curated review package. Annotated-only diagnostic; cluster/family names are hypotheses, not truth labels.
')
manifest=[]
for p in sorted(UPLOAD.rglob('*')):
    if p.is_file() and p.name!='upload_manifest.csv':
        manifest.append({'relative_path':str(p.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'upload_file_list.txt').write_text('
'.join(sorted(x['relative_path'] for x in manifest))+'
')
zip_path=OUT/'Notebook59R3_151_Page_Retained_Morphology_Audit_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file(): z.write(p,p.relative_to(UPLOAD))
with zipfile.ZipFile(zip_path,'r') as z: require(z.testzip() is None,'ZIP integrity test failed.')
atomic_json(OUT/'upload_package_identity.json',{'zip_path':str(zip_path),'zip_sha256':sha256_file(zip_path),'zip_bytes':zip_path.stat().st_size,'files':len(manifest)})
print('Upload ZIP:',zip_path)


## Review order

1. `executive_summary.txt`
2. `tables/special_family_summary.csv`
3. `tables/cluster_profiles.csv`
4. `tables/top_right_recurrence_fingerprints.csv`
5. `contact_sheets/family_top_right_identifier_candidate.png`
6. `contact_sheets/family_long_glyph_candidate.png`
7. cluster representative sheets
8. `tables/failure_family_treatment_hypotheses.csv`

The top-right detector is deliberately positional and diagnostic. It tests recurrence of first-page part/catalogue identifiers such as `D32`; it must not become a production removal shortcut without authoritative labelling and validation. The long-glyph detector tests whether elongated strokes and arrows form a separate family that may require a larger spatial context than 33×33.
